# Sample QC Report with singlet.load_dir()

This notebook demonstrates how `singlet.load_dir()` gives you a complete,
analysis-ready AnnData with all QC metrics pre-computed by singlify.
No manual file parsing needed — one function call gives you everything.

**Sample**: GSM3573650 (GSE125416, Homo sapiens, 10x Chromium v3, PBMC)

In [1]:
import singlet
import numpy as np

# One call loads everything
adata = singlet.load_dir(
    '/mnt/projects/debruinz_project/singlify_pipeline/quant/scrna/GSE125/GSE125416/GSM3573650'
)

print(f'Matrix: {adata.n_obs:,} cells × {adata.n_vars:,} genes')
print(f'\nPer-cell metrics (adata.obs):')
for col in adata.obs.columns:
    print(f'  • {col}')
print(f'\nSample-level metadata (adata.uns):')
for key in adata.uns:
    val = adata.uns[key]
    if isinstance(val, dict):
        print(f'  • {key}: {list(val.keys())[:5]}...')
    else:
        print(f'  • {key}: {val}')

Matrix: 75,420 cells × 38,606 genes

Per-cell metrics (adata.obs):
  • total_counts
  • total_umis
  • total_genes
  • mt_pct
  • ribo_pct
  • intronic_pct
  • doublet_score
  • is_doublet
  • phase
  • s_score
  • g2m_score

Sample-level metadata (adata.uns):
  • geo_source_name: "
  • geo_title: "
  • gse_id: "
  • gsm_id: "
  • modality: "
  • organism: "
  • pipeline_date: 2026-04-14
  • protocol: 10x-3p-v2
  • read_count: 150756402
  • singlify_version: 0.3.0
  • srr_ids: ["SRR8479160"]
  • taxon_id: 0
  • ancestry: ['ancestry', 'confidence', 'n_informative_aims', 'n_covered_aims', 'aim_panel_size']...
  • singlify_dir: /mnt/projects/debruinz_project/singlify_pipeline/quant/scrna/GSE125/GSE125416/GSM3573650


In [2]:
# Quick QC summary
print('═' * 50)
print('SAMPLE QC REPORT')
print('═' * 50)
print(f'  Cells:         {adata.n_obs:>10,}')
print(f'  Genes:         {adata.n_vars:>10,}')
print(f'  Median UMIs:   {adata.obs["total_umis"].median():>10,.0f}')
print(f'  Median genes:  {adata.obs["total_genes"].median():>10,.0f}')
print(f'  Median MT%:    {adata.obs["mt_pct"].median():>10.2f}%')
print(f'  Median ribo%:  {adata.obs["ribo_pct"].median():>10.2f}%')
print(f'  Intronic%:     {adata.obs["intronic_pct"].median():>10.2f}%')
print(f'  Doublet rate:  {adata.obs["is_doublet"].mean():>10.1%}')
print(f'  Cell cycle:    G1={adata.obs["phase"].eq("G1").mean():.0%} '
      f'S={adata.obs["phase"].eq("S").mean():.0%} '
      f'G2M={adata.obs["phase"].eq("G2M").mean():.0%}')
anc = adata.uns.get('ancestry', {})
if anc:
    print(f'  Ancestry:      {anc["ancestry"]} (confidence={anc["confidence"]})')
print('═' * 50)

══════════════════════════════════════════════════
SAMPLE QC REPORT
══════════════════════════════════════════════════
  Cells:             75,420
  Genes:             38,606
  Median UMIs:          228
  Median genes:         182
  Median MT%:          0.71%
  Median ribo%:       36.14%
  Intronic%:           4.78%
  Doublet rate:       13.8%
  Cell cycle:    G1=92% S=2% G2M=5%
  Ancestry:      EUR (confidence=1.0)
══════════════════════════════════════════════════


In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Row 1: Count distributions
axes[0,0].hist(np.log10(adata.obs['total_umis']+1), bins=50, color='#3b82f6', edgecolor='white')
axes[0,0].axvline(np.log10(adata.obs['total_umis'].median()+1), color='red', linestyle='--')
axes[0,0].set_xlabel('log10(UMIs)'); axes[0,0].set_ylabel('Cells')
axes[0,0].set_title(f'UMI Distribution (median={adata.obs["total_umis"].median():,.0f})')

axes[0,1].hist(adata.obs['total_genes'], bins=50, color='#22c55e', edgecolor='white')
axes[0,1].axvline(adata.obs['total_genes'].median(), color='red', linestyle='--')
axes[0,1].set_xlabel('Genes'); axes[0,1].set_ylabel('Cells')
axes[0,1].set_title(f'Gene Count (median={adata.obs["total_genes"].median():,.0f})')

axes[0,2].hist(adata.obs['mt_pct'], bins=50, color='#ef4444', edgecolor='white')
axes[0,2].axvline(adata.obs['mt_pct'].median(), color='black', linestyle='--')
axes[0,2].set_xlabel('MT%'); axes[0,2].set_ylabel('Cells')
axes[0,2].set_title(f'Mitochondrial % (median={adata.obs["mt_pct"].median():.2f}%)')

# Row 2: QC relationships
sub = adata[np.random.choice(adata.n_obs, min(5000, adata.n_obs), replace=False)]
axes[1,0].scatter(sub.obs['total_umis'], sub.obs['total_genes'], alpha=0.2, s=3, c='#3b82f6')
axes[1,0].set_xlabel('UMIs'); axes[1,0].set_ylabel('Genes')
axes[1,0].set_title('UMIs vs Genes')

axes[1,1].scatter(sub.obs['total_umis'], sub.obs['mt_pct'], alpha=0.2, s=3, c='#ef4444')
axes[1,1].set_xlabel('UMIs'); axes[1,1].set_ylabel('MT%')
axes[1,1].set_title('UMIs vs MT%')

# Cell cycle pie
phases = adata.obs['phase'].value_counts()
axes[1,2].pie(phases.values, labels=phases.index, autopct='%1.1f%%',
              colors=['#3b82f6', '#ef4444', '#22c55e'])
axes[1,2].set_title('Cell Cycle Phases')

plt.suptitle('singlify QC Report — GSM3573650 (75,420 cells)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('qc_report.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: qc_report.png')

Saved: qc_report.png


In [4]:
# Filter and compare pre/post
print('Filtering pipeline:')
print(f'  Start: {adata.n_obs:,} cells')

# Remove doublets
clean = adata[~adata.obs['is_doublet'].astype(bool)].copy()
print(f'  After doublet removal: {clean.n_obs:,} cells (-{adata.n_obs - clean.n_obs:,})')

# Remove high-MT cells
clean = clean[clean.obs['mt_pct'] < 20].copy()
print(f'  After MT<20% filter: {clean.n_obs:,} cells')

# Remove low-gene cells  
clean = clean[clean.obs['total_genes'] >= 200].copy()
print(f'  After genes≥200 filter: {clean.n_obs:,} cells')

print(f'\n  Final: {clean.n_obs:,} cells ({clean.n_obs/adata.n_obs:.1%} retained)')
print(f'  Removed: {adata.n_obs - clean.n_obs:,} cells ({(adata.n_obs - clean.n_obs)/adata.n_obs:.1%})')

Filtering pipeline:
  Start: 75,420 cells
  After doublet removal: 63,981 cells (-11,439)


  After MT<20% filter: 63,828 cells


  After genes≥200 filter: 9,368 cells

  Final: 9,368 cells (12.4% retained)
  Removed: 66,052 cells (87.6%)


/mnt/home/debruinz/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


## What load_dir() gives you

| Data | Source File | AnnData Location |
|------|------------|------------------|
| Count matrix | gene_counts.1pz | adata.X |
| Gene names | gene_expression.tsv | adata.var_names |
| Ensembl IDs | gene_expression.tsv | adata.var['gene_id'] |
| Barcodes | auto_barcodes.tsv | adata.obs_names |
| UMI/gene counts | cell_qc_metrics.tsv | adata.obs['total_umis', 'total_genes'] |
| MT/ribo % | cell_qc_metrics.tsv | adata.obs['mt_pct', 'ribo_pct'] |
| Intronic % | cell_qc_metrics.tsv | adata.obs['intronic_pct'] |
| Doublet score | doublet_scores.tsv | adata.obs['doublet_score', 'is_doublet'] |
| Cell cycle | cell_cycle_scores.tsv | adata.obs['phase', 's_score', 'g2m_score'] |
| Ancestry | ancestry_call.json | adata.uns['ancestry'] |

All pre-computed by singlify — no additional analysis needed before filtering.